In [ ]:
import os
import ssl
import zipfile
import shutil
import csv
import urllib.request

ssl._create_default_https_context = ssl._create_unverified_context

def _baixar_remuneracao_media_docentes() -> None:
    os.makedirs("data/raw/tmp", exist_ok=True)

    # fonte: https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais
    def _fazer_download(arquivo: str, destino: str, tentativa: int = 1) -> None:
        url = "https://download.inep.gov.br/informacoes_estatisticas/indicadores_educacionais"
        try:
            urllib.request.urlretrieve(url + arquivo, destino)
        except Exception as e:
            if tentativa <= 5:
                _fazer_download(arquivo, destino, tentativa+1)
                return
            print(f"Erro ao tentar fazer download ({url}{arquivo}): {e}")

    arquivos = {
        "2014": "/2014/remuneracao_media_docentes/Remuneracao_docentes_Brasil_Regioes_UFs_2014.zip",
        "2015": "/2015/remuneracao_media_docentes/Remuneracao_docentes_Brasil_Regioes_UFs_2015.zip",
        "2016": "/2016/remuneracao_media_docentes/Remuneracao_docentes_Brasil_Regioes_UFs_2016.zip",
        "2017": "/2017/remuneracao_media_docentes/Remuneracao_docentes_Brasil_Regioes_UFs_2017.zip",
        "2018": "/2018/remuneracao_media_docentes/remuneracao_docentes_uf_2018.zip",
        "2019": "/2019/remuneracao_media_docentes/remuneracao_docentes_uf_2019.zip",
        "2020": "/2020/remuneracao_media_docentes/remuneracao_docentes_uf_2020.zip",
        "2021": "/2021/remuneracao_docentes_brasil_regioes_ufs_2021.zip"
    }

    for ano, arquivo in arquivos.items():
        _fazer_download(arquivo, f"data/raw/tmp/remunecarao-media-docentes-{ano}.zip")

def _extrair_xlsx() -> None:
    os.makedirs("data/raw/remuneracao-media-docentes", exist_ok=True)

    for arquivo in sorted(os.listdir("data/raw/tmp")):
        ano = "".join(c for c in os.path.basename(arquivo) if c.isdigit())
        zip = zipfile.ZipFile(f"data/raw/tmp/{arquivo}", 'r')
        for item in zip.namelist():
            if item.endswith(".xlsx") and zip.getinfo(item).file_size >= 40 * 1024:
                fonte = zip.open(item)
                destino = open(f"data/raw/remuneracao-media-docentes/{ano}.xlsx", "wb")
                destino.write(fonte.read())

def _excluir_tmp() -> None:
    shutil.rmtree("data/raw/tmp")

def _salvar_dados_brutos_ipca() -> None:
    os.makedirs("data/raw", exist_ok=True)

    # fonte: https://www.ipeadata.gov.br/ExibeSerie.aspx?serid=1410807112&module=M
    dados = [
        [2015, 10.67],
        [2016, 6.29],
        [2017, 2.95],
        [2018, 3.75],
        [2019, 4.31],
        [2020, 4.52],
        [2021, 10.06]
    ]

    with open(
        'data/raw/ipca.csv',
        mode='w',
        newline='',
        encoding='utf-8'
    ) as f:
        writer = csv.writer(f)
        writer.writerow(['ano', 'valor'])
        writer.writerows(dados)

In [ ]:
_baixar_remuneracao_media_docentes()

In [ ]:
_extrair_xlsx()

In [ ]:
_excluir_tmp()

In [ ]:
_salvar_dados_brutos_ipca()

In [ ]:
%pip install pandas deltalake openpyxl pyarrow

In [ ]:
import os
import pandas
from deltalake import write_deltalake

def _ingerir_remuneracao_media_docentes() -> None:
    arquivos = {
        "2014": "data/raw/remuneracao-media-docentes/2014.xlsx",
        "2015": "data/raw/remuneracao-media-docentes/2015.xlsx",
        "2016": "data/raw/remuneracao-media-docentes/2016.xlsx",
        "2017": "data/raw/remuneracao-media-docentes/2017.xlsx",
        "2018": "data/raw/remuneracao-media-docentes/2018.xlsx",
        "2019": "data/raw/remuneracao-media-docentes/2019.xlsx",
        "2020": "data/raw/remuneracao-media-docentes/2020.xlsx",
        "2021": "data/raw/remuneracao-media-docentes/2021.xlsx"
    }

    ufs = [
        "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", 
        "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
    ]

    dados = []
    for ano, arquivo in arquivos.items():
        df = pandas.read_excel(arquivo, header=None, dtype=str)
        for uf in ufs:
            condicao = (df[2].str.upper() == uf) & (df[4].str.upper() == 'TOTAL')
            [
                dados.append([ano, uf, *_linha])
                for _linha
                in df.loc[condicao][[3, 13]].values.tolist()
            ]

    os.makedirs("data/bronze/remuneracao-media-docentes", exist_ok=True)
    df = pandas.DataFrame(dados, columns=['ano', 'uf', 'dependencia_administrativa', 'valor'])
    df['valor'] = df['valor'].replace(['*', 'd'], None)
    write_deltalake(f"data/bronze/remuneracao-media-docentes", df, mode="overwrite")

def _ingerir_ipca() -> None:
    df = pandas.read_csv('data/raw/ipca.csv', dtype=str)
    os.makedirs("data/bronze", exist_ok=True)
    df.to_parquet(f"data/bronze/ipca.snappy.parquet", engine="pyarrow", compression="snappy")

In [ ]:
_ingerir_remuneracao_media_docentes()

In [ ]:
_ingerir_ipca()

In [ ]:
%pip install duckdb

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM 'data/bronze/ipca.snappy.parquet'").show()

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")
con.sql(
    "SELECT * FROM delta_scan('data/bronze/remuneracao-media-docentes')"
).show()

In [ ]:
%pip install dbt-duckdb

In [ ]:
%cd dbt
!dbt deps
%cd ..

In [ ]:
%cd dbt
!dbt docs generate
!dbt docs serve
%cd ..

In [ ]:
import os

os.makedirs("data/silver", exist_ok=True)

In [ ]:
%cd dbt
!dbt build --select silver
%cd ..

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")
con.sql(
    "SELECT * FROM delta_scan('data/bronze/remuneracao-media-docentes') "
    "WHERE valor IS NULL"
).show()

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")

con.sql(
    "SELECT * FROM delta_scan('data/bronze/remuneracao-media-docentes', version=0) "
    "WHERE ano = 2014 and uf = 'RJ' OR ano = 2021 and uf = 'RO'"
).show()

In [ ]:
from deltalake import DeltaTable, write_deltalake

df = DeltaTable("data/bronze/remuneracao-media-docentes").to_pandas()

df.loc[
    (df['ano'] == "2014") & 
    (df['uf'] == 'RJ') & 
    (df['dependencia_administrativa'].isin(['Estadual', 'Pública'])),
    'valor'
] = df.loc[
    (df['ano'] == "2014") & 
    (df['uf'] == 'RJ') & 
    (df['dependencia_administrativa'] == 'Privada')
]['valor'].iloc[0]

df.loc[
    (df['ano'] == "2021") & 
    (df['uf'] == 'RO') & 
    (df['dependencia_administrativa'] == 'Estadual'),
    'valor'
] = df.loc[
    (df['ano'] == "2021") & 
    (df['uf'] == 'RO') & 
    (df['dependencia_administrativa'] == 'Pública')
]['valor'].iloc[0]

write_deltalake(f"data/bronze/remuneracao-media-docentes", df, mode="overwrite")

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")
con.sql(
    "SELECT * FROM delta_scan('data/bronze/remuneracao-media-docentes') "
    "WHERE valor IS NULL"
).show()

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")

print("Dados da versão 0:")
con.sql(
    "SELECT * FROM delta_scan('data/bronze/remuneracao-media-docentes', version=0) "
    "WHERE ano = 2014 and uf = 'RJ' OR ano = 2021 and uf = 'RO'"
).show()

print("Dados da versão 1:")
con.sql(
    "SELECT * FROM delta_scan('data/bronze/remuneracao-media-docentes', version=1) "
    "WHERE ano = 2014 and uf = 'RJ' OR ano = 2021 and uf = 'RO'"
).show()

In [ ]:
%cd dbt
!dbt build --select silver
%cd ..

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM 'data/silver/stg_tempo.parquet'").show()

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM 'data/silver/stg_uf.parquet'").show()

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM 'data/silver/stg_dependencia_administrativa.parquet'").show()

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM 'data/silver/stg_ipca.parquet'").show()

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM 'data/silver/stg_remuneracao_media_docentes.parquet'").show()

In [ ]:
import os

os.makedirs("data/gold", exist_ok=True)

In [ ]:
%cd dbt
!dbt build --select +gold
%cd ..

In [ ]:
import duckdb

print("Dimensão tempo:")
duckdb.sql("SELECT * FROM 'data/gold/dim_tempo.parquet'").show()

In [ ]:
import duckdb

print("Dimensão uf:")
duckdb.sql("SELECT * FROM 'data/gold/dim_uf.parquet'").show()

In [ ]:
import duckdb

print("Dimensão uf:")
duckdb.sql("SELECT * FROM 'data/gold/dim_dependencia_administrativa.parquet'").show()

In [ ]:
%pip install pygwalker

In [ ]:
import duckdb
import pygwalker as pyg
from IPython.display import display, HTML

df_maiores_remuneracoes = duckdb.sql(
    "SELECT * FROM 'data/gold/fato_maiores_remuneracoes.parquet'"
).df()

walker = pyg.walk(df_maiores_remuneracoes)
display(HTML(walker.to_html()))

In [ ]:
import duckdb
import pygwalker as pyg
from IPython.display import display, HTML

df_menores_remuneracoes = duckdb.sql(
    "SELECT * FROM 'data/gold/fato_menores_remuneracoes.parquet'"
).df()

walker = pyg.walk(df_menores_remuneracoes)
display(HTML(walker.to_html()))

In [ ]:
import duckdb
import pygwalker as pyg
from IPython.display import display, HTML

df_diferenca_remuneracoes_publica_privada = duckdb.sql(
    "SELECT * FROM 'data/gold/fato_diferenca_remuneracoes_publica_privada.parquet'"
).df()

walker = pyg.walk(df_diferenca_remuneracoes_publica_privada)
display(HTML(walker.to_html()))

In [ ]:
import duckdb
import pygwalker as pyg
from IPython.display import display, HTML

df_reajuste_salarial_vs_ipca = duckdb.sql(
    "SELECT * FROM 'data/gold/fato_reajuste_salarial_vs_ipca.parquet'"
).df()

walker = pyg.walk(df_reajuste_salarial_vs_ipca)
display(HTML(walker.to_html()))